<a href="https://www.kaggle.com/code/avikdas567/deep-past-challenge-translate-akkadian-to-english?scriptVersionId=292057272" target="_blank"><img align="left" alt="Kaggle" title="Open in Kaggle" src="https://kaggle.com/static/images/open-in-kaggle.svg"></a>

In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

/kaggle/input/deep-past-initiative-machine-translation/sample_submission.csv
/kaggle/input/deep-past-initiative-machine-translation/bibliography.csv
/kaggle/input/deep-past-initiative-machine-translation/publications.csv
/kaggle/input/deep-past-initiative-machine-translation/Sentences_Oare_FirstWord_LinNum.csv
/kaggle/input/deep-past-initiative-machine-translation/OA_Lexicon_eBL.csv
/kaggle/input/deep-past-initiative-machine-translation/eBL_Dictionary.csv
/kaggle/input/deep-past-initiative-machine-translation/train.csv
/kaggle/input/deep-past-initiative-machine-translation/test.csv
/kaggle/input/deep-past-initiative-machine-translation/published_texts.csv
/kaggle/input/deep-past-initiative-machine-translation/resources.csv


In [2]:
# ============================================================
# Deep Past Challenge - Akkadian → English
# ============================================================

import re
import pandas as pd

# ---------------------------
# Load data
# ---------------------------
DATA_DIR = "/kaggle/input/deep-past-initiative-machine-translation"
test_df = pd.read_csv(f"{DATA_DIR}/test.csv")

print("Test size:", len(test_df))

# ---------------------------
# Cleaning function
# ---------------------------
def clean_transliteration(text):
    if pd.isna(text):
        return ""

    text = text.lower()

    # Remove scribal notation
    text = re.sub(r"[!?/:]", " ", text)
    text = re.sub(r"[\[\]˹˺]", "", text)
    text = re.sub(r"\b\d+'*\b", " ", text)

    # Gaps
    text = re.sub(r"\.{2,}", " [missing text] ", text)
    text = re.sub(r"\bx\b", " [gap] ", text)

    # Determinatives → hint tokens
    text = re.sub(r"\{d\}", " god ", text)
    text = re.sub(r"\{ki\}", " place ", text)
    text = re.sub(r"\{m\}", " man ", text)
    text = re.sub(r"\{f\}", " woman ", text)
    text = re.sub(r"\{[^}]+\}", " ", text)

    text = re.sub(r"\s+", " ", text).strip()
    return text

# ---------------------------
# Pseudo-translation
# ---------------------------
def pseudo_translate(text):
    if not text:
        return ""

    # Very common Akkadian epistolary patterns
    if text.startswith("umma"):
        return "Thus says " + text.replace("umma", "").strip()

    if "ana" in text:
        return "To " + text.replace("ana", "").strip()

    # Default: readable paraphrase
    return "Regarding the matter: " + text

# ---------------------------
# Generate translations
# ---------------------------
translations = []

for t in test_df["transliteration"]:
    cleaned = clean_transliteration(t)
    translated = pseudo_translate(cleaned)
    translations.append(translated)

# ---------------------------
# Submission
# ---------------------------
submission = pd.DataFrame({
    "id": test_df["id"],
    "translation": translations
})

submission.to_csv("submission.csv", index=False)

print("submission.csv created successfully")
submission.head()

Test size: 4
submission.csv created successfully


,id,translation
0,0,Regarding the matter: um-ma kà-ru-um kà-ni-ia-...
1,1,Regarding the matter: i-na mup-pì-im aa a-lim(...
2,2,Regarding the matter: ki-ma mup-pì-ni ta-áa-me...
3,3,Regarding the matter: me-+e-er mup-pì-ni a-na ...
